# Task #2 - Initial Data Analysis

This notebook checks the bearing fault dataset for missing values, invalid values, and unusual signal patterns. It does not remove samples automatically. Instead, it flags signals that may need a quick review before the team moves on to standardization and modeling.

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import scipy.io

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
print("Reading from:", DATA_DIR.resolve())

Reading from: /Users/chikaosolunnadozie/Desktop/projects/MathWorks/data


In [2]:
def load_split(split):
    mat = scipy.io.loadmat(DATA_DIR / f"{split}.mat")
    X = mat[f"{split}Data"]
    y = [str(label[0]) for label in mat[f"{split}Labels"].flatten()]
    return X, y

splits = {}
for split in ["train", "val", "test"]:
    X, y = load_split(split)
    splits[split] = {"X": X, "y": y}

In [3]:
summary_rows = []

for split, info in splits.items():
    X = info["X"]
    y = info["y"]

    summary_rows.append({
        "split": split,
        "signals": X.shape[0],
        "samples_per_signal": X.shape[1],
        "dtype": str(X.dtype),
        "class_counts": dict(Counter(y)),
        "nan_count": int(np.isnan(X).sum()),
        "inf_count": int(np.isinf(X).sum()),
        "signal_mean": float(X.mean()),
        "signal_std": float(X.std()),
        "signal_min": float(X.min()),
        "signal_max": float(X.max()),
    })

summary = pd.DataFrame(summary_rows).set_index("split")
summary

,signals,samples_per_signal,dtype,class_counts,nan_count,inf_count,signal_mean,signal_std,signal_min,signal_max
split,,,,,,,,,,
train,393,5000,float64,"{'InnerRaceFault': 131, 'OuterRaceFault': 131,...",0,0,-0.153787,1.287054,-45.97387,39.31665
val,27,5000,float64,"{'InnerRaceFault': 9, 'OuterRaceFault': 9, 'No...",0,0,-0.158776,1.230148,-23.54578,23.72603
test,102,5000,float64,"{'InnerRaceFault': 34, 'OuterRaceFault': 34, '...",0,0,-0.151315,1.291981,-26.88561,27.49819


## Notes on the Results

The expected class counts are already shown in the data documentation. This step mainly confirms that the loaded data is clean and that the splits look balanced.

If `nan_count` and `inf_count` are both 0, the dataset is clean enough to move forward.

In [4]:
signal_stats_rows = []

for split, info in splits.items():
    X = info["X"]
    y = info["y"]

    signal_mean = X.mean(axis=1)
    signal_std = X.std(axis=1)
    signal_max = np.max(np.abs(X), axis=1)
    p25 = np.percentile(X, 25, axis=1)
    p75 = np.percentile(X, 75, axis=1)
    iqr = p75 - p25

    signal_stats_rows.extend(
        [
            {
                "split": split,
                "label": label,
                "mean": float(mean),
                "std": float(std),
                "max_abs": float(max_abs),
                "iqr": float(iqr_val),
            }
            for label, mean, std, max_abs, iqr_val in zip(y, signal_mean, signal_std, signal_max, iqr)
        ]
    )

signal_stats = pd.DataFrame(signal_stats_rows)

Q1 = signal_stats["std"].quantile(0.25)
Q3 = signal_stats["std"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

signal_stats["std_outlier_flag"] = (signal_stats["std"] < lower) | (signal_stats["std"] > upper)
signal_stats["max_abs_outlier_flag"] = (signal_stats["max_abs"] > signal_stats["max_abs"].quantile(0.99))

signal_stats.head()

,split,label,mean,std,max_abs,iqr,std_outlier_flag,max_abs_outlier_flag
0,train,InnerRaceFault,-0.211336,2.256738,23.38188,1.128970,False,False
1,train,InnerRaceFault,-0.219043,1.695879,20.11251,1.018128,False,False
2,train,InnerRaceFault,-0.203583,1.958869,22.91833,1.027782,False,False
3,train,InnerRaceFault,-0.233934,1.511303,11.96552,0.996153,False,False
4,train,InnerRaceFault,-0.242097,1.687088,18.61170,1.129188,False,False


In [5]:
flagged = signal_stats[
    signal_stats["std_outlier_flag"] | signal_stats["max_abs_outlier_flag"]
]

flagged = flagged.sort_values(["split", "std"], ascending=[True, False])
flagged.head(20)

,split,label,mean,std,max_abs,iqr,std_outlier_flag,max_abs_outlier_flag
75,train,InnerRaceFault,-0.229835,2.242398,42.97734,0.978178,False,True
38,train,InnerRaceFault,-0.226180,2.123774,45.97387,1.017685,False,True
128,train,InnerRaceFault,-0.211764,2.032892,36.96610,0.987881,False,True
121,train,InnerRaceFault,-0.243137,2.031893,32.18819,0.986641,False,True
95,train,InnerRaceFault,-0.229664,1.950556,41.83310,0.990545,False,True
64,train,InnerRaceFault,-0.235580,1.830381,32.89919,0.969776,False,True


## What to do with the flagged signals

For this project, do not remove flagged signals just because they look different. Many of them may be real fault examples.

Use the flagged list to review a few signals by hand. Remove only samples that are truly corrupted, such as those with `NaN`, `Inf`, or broken values.